# Resume–Job Match Scorer (Google Colab)

Run each cell in order. You'll upload your resume and paste the job description, then get a match score and improvement tips.

## Step 1: Install dependencies

Run this cell once. It installs PyPDF2 and sentence-transformers (~80MB model download on first run).

In [ ]:
!pip install -q PyPDF2 sentence-transformers numpy
print("Done.")

## Step 2: Load libraries and model

Run this cell once. It loads the embedding model (first time may download ~80MB).

In [ ]:
import io
import re
from sentence_transformers import SentenceTransformer
import PyPDF2
import numpy as np

print("Loading embedding model (first time may take a minute)...")
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Ready.")

## Step 3: Upload resume and paste job description

Run this cell. A file picker will appear—upload your **PDF** resume. Then paste the job description (multiple lines OK). When finished, type **END** on a new line and press Enter.

In [ ]:
from google.colab import files

uploaded = files.upload()  # Click "Choose Files" and select your resume PDF
if not uploaded:
    raise SystemExit("No file uploaded. Run the cell again and select a PDF.")

resume_path = list(uploaded.keys())[0]
with open(resume_path, "rb") as f:
    resume_bytes = f.read()

# Extract text from PDF (no function - inline only)
if resume_path.lower().endswith(".pdf"):
    reader = PyPDF2.PdfReader(io.BytesIO(resume_bytes))
    resume_text = ""
    for page in reader.pages:
        resume_text += page.extract_text() or ""
    resume_text = resume_text.strip() or ""
else:
    resume_text = resume_bytes.decode("utf-8", errors="replace")

if not resume_text.strip():
    raise SystemExit("Could not extract text from the file.")

print("Resume uploaded and text extracted.")
print("Paste the job description below (multiple lines OK). When done, type END on a new line and press Enter:")
lines = []
while True:
    line = input()
    if line.strip().upper() == "END":
        break
    lines.append(line)
job_description = "\n".join(lines).strip()

if not job_description:
    raise SystemExit("Job description is required. Run the cell again and paste the job, then type END.")

job_text = job_description
print("Job description saved.")

## Step 4: Get match score and tips

Run this cell to compute the score and see improvement tips.

In [ ]:
# Keyword score (word overlap)
resume_lower = resume_text.lower()
job_lower = job_text.lower()
stop = {"the", "and", "for", "with", "or", "to", "in", "of", "a", "an", "is", "on", "at"}
job_words = set(re.findall(r"\b[a-z]{3,}\b", job_lower)) - stop
resume_words = set(re.findall(r"\b[a-z]{3,}\b", resume_lower))
kw = (len(job_words & resume_words) / len(job_words) * 100) if job_words else 50.0
kw = min(100.0, kw)

# Semantic score (embedding similarity)
r = resume_text[:4000] if len(resume_text) > 4000 else resume_text
j = job_text[:4000] if len(job_text) > 4000 else job_text
embs = model.encode([r, j])
sim = np.dot(embs[0], embs[1]) / (np.linalg.norm(embs[0]) * np.linalg.norm(embs[1]) + 1e-9)
sem = (sim + 1) * 50

# Final score
score = round(0.4 * kw + 0.6 * sem, 1)

# Build tips (no function - inline only)
tips = []
job_words_4 = set(re.findall(r"\b[a-z]{4,}\b", job_lower))
resume_words_4 = set(re.findall(r"\b[a-z]{4,}\b", resume_lower))
missing = job_words_4 - resume_words_4
if score < 70 and missing:
    sample = list(missing)[:5]
    tips.append(f"Consider adding these terms from the job: {', '.join(sample)}.")
if score < 60:
    tips.append("Expand your experience section to better align with the job.")
if "experience" in job_lower and "experience" not in resume_lower and "work" not in resume_lower:
    tips.append("Add a clear Experience or Work History section.")
if not tips:
    tips.append("Your resume is well aligned. Consider tailoring the summary to this role.")
tips = tips[:5]

# Show results
print("=" * 50)
print("  MATCH SCORE:", score, "/ 100")
print("=" * 50)
print()
print("Improvement tips:")
for i, tip in enumerate(tips, 1):
    print(f"  {i}. {tip}")
print()
if score >= 70:
    print("Overall: Strong fit for this role.")
else:
    print("Overall: Consider the tips above to better align your resume.")